# 04 — Fairness Audit & Bias Mitigation

Audits the tuned XGBoost model's predictions for disparate impact across `SEX` and `AGE_GROUP` (Fairlearn: demographic parity ratio/difference, equalized-odds difference, four-fifths-rule convention), then applies reweighing (Kamiran & Calders, 2012) on `SEX` and re-audits both attributes post-mitigation.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import joblib
from sklearn.model_selection import train_test_split

from dac.config import CONFIG
from dac.data.loader import load_uci_credit
from dac.fairness.audit import audit_fairness
from dac.fairness.mitigation import compute_reweighing_weights
from dac.features.engineering import build_preprocessor, engineer_uci_credit_features, split_feature_columns

uci_cfg = CONFIG["data"]["uci_credit"]
protected_attributes = uci_cfg["protected_attributes"]
df, _ = load_uci_credit()
df = engineer_uci_credit_features(df)
exclude_cols = [uci_cfg["id_col"], *protected_attributes]
numeric_cols, categorical_cols = split_feature_columns(df, uci_cfg["target_col"], exclude_cols)
X = df[numeric_cols + categorical_cols]
y = df[uci_cfg["target_col"]]
sensitive = df[protected_attributes]
X_train, X_test, y_train, y_test, sens_train, sens_test = train_test_split(
    X, y, sensitive, test_size=CONFIG["split"]["test_size"], random_state=CONFIG["seed"], stratify=y
)
pipeline = joblib.load(CONFIG["paths"]["models_dir"] / "xgboost_tuned.joblib")
y_pred = (pipeline.predict_proba(X_test)[:, 1] >= 0.5).astype(int)

## Pre-mitigation audit

In [ ]:
pre = {}
for attr in protected_attributes:
    pre[attr] = audit_fairness(
        y_true=y_test.to_numpy(), y_pred=y_pred, sensitive_features=sens_test[attr],
        model_name="xgboost_tuned_pre_mitigation", attribute_name=attr,
        figures_dir=CONFIG["paths"]["figures_dir"] / "uci_credit" / "fairness",
        metrics_dir=CONFIG["paths"]["metrics_dir"] / "fairness",
        favorable_label=CONFIG["fairness"]["favorable_label"],
    )
pre

## Reweighing mitigation + re-audit

In [ ]:
from dac.models.train import build_xgboost_pipeline, compute_scale_pos_weight

mitigation_attr = CONFIG["fairness"]["mitigation_attribute"]
weights = compute_reweighing_weights(y_train, sens_train[mitigation_attr])

preprocessor = build_preprocessor(numeric_cols, categorical_cols)
mitigated_pipeline = build_xgboost_pipeline(preprocessor, scale_pos_weight=compute_scale_pos_weight(y_train))
mitigated_pipeline.fit(X_train, y_train, clf__sample_weight=weights)
y_pred_mitigated = (mitigated_pipeline.predict_proba(X_test)[:, 1] >= 0.5).astype(int)

In [ ]:
post = {}
for attr in protected_attributes:
    post[attr] = audit_fairness(
        y_true=y_test.to_numpy(), y_pred=y_pred_mitigated, sensitive_features=sens_test[attr],
        model_name="xgboost_tuned_post_mitigation", attribute_name=attr,
        figures_dir=CONFIG["paths"]["figures_dir"] / "uci_credit" / "fairness",
        metrics_dir=CONFIG["paths"]["metrics_dir"] / "fairness",
        favorable_label=CONFIG["fairness"]["favorable_label"],
    )
post

`scripts/run_pipeline.py` runs this same reweighing recipe across all four tuned model families (not just XGBoost) — that cross-model comparison is RQ4; see `reports/model_performance/run_summary.json`.